# Install Dependencies

In [ ]:
%pip install -U \
  "streamlit>=1.39.0" \
  "huggingface-hub>=1.23.0" \
  "transformers>=4.40.0" \
  "accelerate>=0.28.0" \
  "llama-index-core>=0.11.0" \
  "llama-index-embeddings-huggingface>=0.3.0" \
  "llama-index-readers-wikipedia>=0.6.0" \
  "llama-index-vector-stores-pinecone>=0.3.0" \
  "pinecone[grpc]>=5.0.0" \
  "wikipedia>=1.4.0"

# Run RAG System

In [ ]:
%%writefile rag_chatbot.py

# Imports necessary modules, libraries
import os, wikipedia
from google.colab import userdata
from llama_index.embeddings.huggingface import HuggingFaceEmbedding # llama_index.embeddings.huggingface_openvino OpenVINOEmbedding
# Imports storage modules to cache data (locally)
from llama_index.core import VectorStoreIndex, StorageContext, Settings
# Imports wikipedia reader to retrieve data dynamically from Wikipedia
from llama_index.readers.wikipedia import WikipediaReader
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter
from llama_index.core.ingestion import IngestionPipeline, DocstoreStrategy
import re
from pinecone.grpc import PineconeGRPC
from pinecone import ServerlessSpec
from dotenv import load_dotenv
wikipedia.set_user_agent("CMSC437-RAG-Project/1.0 (balajia@vcu.edu)")
# Import streamlit for frontend
import streamlit as st
# Imports transformers library to access inference model (Qwen 3 4B)
from transformers import AutoTokenizer, AutoModelForCausalLM



# -------------------------------------------------------------------
# Load LLM model: Loads the models ONCE across all user sessions and app reruns
# -------------------------------------------------------------------

@st.cache_resource
def loadModel():

    # hf_hub.snapshot_download(model_id, local_dir = model_path)
    # llm = OpenVINOGenAILLM(model_path = model_path, device = "GPU")
    # Initializes embedding model (dense embedding approach)
    Settings.embed_model = HuggingFaceEmbedding(model_name = "Qwen/Qwen3-Embedding-0.6B", device = "cuda")
    # Settings.chunk_size = 1000

    modelName = "Qwen/Qwen3-4B"
    tokenizer = AutoTokenizer.from_pretrained(modelName)
    model = AutoModelForCausalLM.from_pretrained(modelName, torch_dtype = "auto", device_map = "auto")

    # llm = ov_genai.LLMPipeline(model_path, device = "GPU")
    return Settings.embed_model, tokenizer, model



# -------------------------------------------------------------------
# ETL (Extract, Transform, Load data. Retrieve & Store data
# -------------------------------------------------------------------


# Retrieval step: Extracts data from Wikipedia
def extractData(pages):

    # If not in cache, the appropriate Wikipedia page is read and loaded to be converted into vector embeddings and then cached
    reader = WikipediaReader()
    data = []

    for page in pages:
        try:
            # Attempt to extract the page data normally
            pageData = reader.load_data(pages = [page])
            for doc in pageData:
                doc.metadata["title"] = page
            data.extend(pageData)
        except wikipedia.exceptions.DisambiguationError as e:
            # Handle the multi-match mapping clash gracefully
            st.warning(f"Wikipedia title ambiguous for '{page}'. Possible options: {e.options[:3]}")

            # Smart Fallback Logic: Try to look for an options match or use the first suggestion
            if e.options:
                fallbackPage = e.options[0] # Grab the first specific disambiguation suggestion
                try:
                    pageData = reader.load_data(pages=[fallbackPage])
                    for doc in pageData:
                        doc.metadata["title"] = page
                    data.extend(pageData)
                except Exception:
                    continue # Skip safely if the fallback fails as well
        except wikipedia.exceptions.PageError:
            st.error(f"Wikipedia page not found for title: '{page}'")
            continue

    return data


# Transforms data (chunking by section)
def transformData(pages, vectorStore):

    # Extracts data from the page title (highest matching title from actual) if not already in Pinecone
    rawData = extractData(pages)

    # 2. Pre-processing: Clean and convert headers to Markdown syntax
    for doc in rawData:
        current_text = doc.text

        # Truncate trailing bibliography blocks cleanly in memory
        for stop_word in ["\n== See also ==", "\n== References ==", "\n== External links =="]:
            if stop_word in current_text:
                current_text = current_text.split(stop_word)[0]

        # Convert ALL levels of MediaWiki headers to ATX Markdown hashes
        current_text = re.sub(r'==== (.*?) ====', r'#### \1', current_text) # Sub-subsections
        current_text = re.sub(r'=== (.*?) ===', r'### \1', current_text)   # Subsections
        current_text = re.sub(r'== (.*?) ==', r'## \1', current_text)     # Main sections

        # Write the final, beautifully cleaned string back to the Document object ONCE
        doc.set_content(current_text)


        # Extracts title or falls back to cleaning up the first chunk text as a label
        title_metadata = doc.metadata.get("title", doc.id_)
        doc.id_ = f"wiki_{str(title_metadata).lower().replace(' ', '_').replace('–', '-')}"


    # Define the pipeline to chunk data and transform into vector embeddings to store in Pinecone
    pipeline = IngestionPipeline(
        transformations = [
            MarkdownNodeParser(),                                   # Step 1: Split by sections
            SentenceSplitter(chunk_size = 512, chunk_overlap = 50), # Step 2: Bound by tokens

            # Step 3: Vector Embedding Generation
            # This will automatically loop through every chunk, send it to the embedding model (Qwen 3 embedding),
            # and attach the resulting vector directly to the node's payload.
            Settings.embed_model
        ],
        vector_store = vectorStore,
        # Updates duplicate documents and runs transformations in LlamaIndex when a document's ID is missing or its content hash has changed
        docstore_strategy = DocstoreStrategy.UPSERTS
    )

    nodes = pipeline.run(documents = rawData)
    return nodes


# Creates connection to Pinecone (vector database) to store data
def loadData():

    try:

        # os.environ['PINECONE_API_KEY'] = apiKey
        pc = PineconeGRPC(api_key = 'INSERT_YOUR_API_KEY_HERE')
        # Index name for unique identifier to create index
        index_name = "cricket-wikipedia-rag"

        # 1. Ensure the index exists, otherwise create it
        existingIndexes = [info["name"] for info in pc.list_indexes()]
        if index_name not in existingIndexes:

            # Create and configure an index (use this to organize the database which stores the vector embeddings)
            pc.create_index(
                name = index_name,
                dimension = 1024, # Must match the dimensions of the embedding model
                spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
            )

        # 2. Establish a connection to the cloud database
        pcIndex = pc.Index(name = index_name)
        # Allows you to store, index, and search through the vector embeddings in Pinecone
        vectorStore = PineconeVectorStore(pinecone_index = pcIndex)


        # Define Cricket scope
        cricketPages = [
            "Cricket", "Laws of Cricket", "Cricket World Cup", "ICC Men's T20 World Cup", "List of Cricket World Cup finals",
            "Test cricket", "One Day International", "Duckworth–Lewis–Stern method",
            "Indian Premier League", "Sachin Tendulkar", "Virat Kohli", "Jasprit Bumrah", "Shane Warne (cricketer)", "Ravichandran Ashwin"
        ]


        # Automatically check Pinecone for each page using metadata tracking
        # This acts as your smart ETL gatekeeper!
        missingPages = []
        for page in cricketPages:
            targetID = f"wiki_{page.lower().replace(' ', '_').replace('–', '-')}"

            # Fetch records directly by ID prefix to avoid zero-vector filtering bugs
            # Since your IngestionPipeline upserts chunks with parent ID prefixes,
            # we check for the base page ID directly.
            fetchResponse = pcIndex.fetch(ids = [targetID])

            # If the ID doesn't exist in the cluster layout, mark it for ingestion
            if not fetchResponse.get("vectors"):
                missingPages.append(page)

        # Process and load missing entities into your vector space on the fly
        if missingPages:
            st.info(f"Syncing missing knowledge map pages into Pinecone: {missingPages}")
            transformData(missingPages, vectorStore)
            st.success("Automated database sync complete!")

        else:
            st.sidebar.success("Database fully synchronized. Reading directly from Pinecone.")

        # Initialize retriever context directly from the database snapshot instantly
        idx = VectorStoreIndex.from_vector_store(vector_store = vectorStore)

    except Exception as e:
        st.error("Couldn't connect to the knowledge database. Please try again shortly.")
        st.stop()

    # return index to access it and retrieve data from the database
    return idx, vectorStore


# -------------------------------------------------------------------
# Streamlit code + LLM implementation (frontend to interact with LLM)
# -------------------------------------------------------------------


# Formats the chats into a text bubble to read user's query and model's response
def text_bubble(text, color, align, text_color):

    persona, message = text.split(":", 1)
    message = message.strip().replace("\n", "<br>")
    html = f"""
        <div style="background-color: {color}; padding: 10px; border-radius: 10px; margin: 10px 0; text-align: left; display: inline-block; max-width: 80%; word-wrap: break-word;">
            <span style="font-size: 16px; font-weight: bold; color: {text_color};">{persona}:</span> <span style="font-size: 16px; font-weight: normal; color: {text_color};">{message}</span>
        </div>
    """

    st.write(html, unsafe_allow_html=True)


# Stores conversations in a container format (like ChatGPT) for user to view history of their chats and LLM answers
def storeChat(chat, answer):

    # Keeps every question/answer pair in one dictionary inside session_state
    # Key = user's question, Value = LLM's answer (plain string)
    # Example: {"What is NLP?": "NLP stands for Natural Language Processing."}
    st.session_state.setdefault("History", {})
    st.session_state["History"][chat] = {"answer": answer}


# LLM leverages the system prompt and context to provide reliable inference
def llmInference(tokenizer, model, question, context):

    # System prompt to make sure LLM is following standard rules with the user's question passed in. Includes few-shot examples to help Qwen format answers
    systemPrompt = f"""
        You are a factual, evidence-based AI assistant. Your task is to answer the user query using ONLY the verified Wikipedia text provided below.

        Use the following context to support your answer:
        <wikipedia_context>
        {context}
        </wikipedia_context>


        Follow the rules below:

        <rules>
        1. Base your answer strictly on the text provided. Find the information even if it's deeply nested.
        2. Respond to the user prompt normally. Do not use, include, or output any special control tokens, markdown code blocks containing ChatML formatting, or raw tags like <|im_start|> or <|im_end|>. End your response naturally with a regular sentence and a period.
        3. You must always respond in English, regardless of the input language.
        4. After your answer, add a line starting with "Evidence:" that quotes the exact sentence from the context to support your answer.
        5. Only attach a specific date to a fact if that exact date appears directly next to that fact in the context. If a fact has no date attached to it in the source text, state it without inventing or borrowing one.
        6. Respond with "I cannot find information about that in the retrieved context." IF the information is genuinely not found on any of the Wikipedia pages given.
        7. Do NOT output your internal thinking, reasoning steps, or assumptions.
        8. If the context describes multiple similar events (e.g. different matches, different years), do not apply a detail from one event to a different event. Match the specific event named in the question exactly.
        </rules>


        Use these examples to format your answers to respond to questions:
        <examples>
            <example1>
                <input> What is the capital of the United States? </input>
                <output>
                The capital of the United States is Washington D.C.
                Evidence: "Washington, D.C., officially the District of Columbia and commonly known as Washington, is the capital city of the United States..."
                </output>
            </example1>

            <example2>
                <input> Who founded Apple Inc? </input>
                <output>
                Apple was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne.
                Evidence: "Founded in 1976 as Apple Computer Company by Steve Jobs, Steve Wozniak and Ronald Wayne..."
                </output>
            </example2>
        </examples>


        Leverage the examples and context above and respond to the user's question:
        <user_question>
        {question}
        </user_question>
        """

    # Initializes answer
    answer = None

    # Set try/except block to send system prompt to Qwen for inference
    try:

        messages = [
            {"role": "user", "content": systemPrompt}
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True,
            tokenize = True,
            enable_thinking = False,
            return_dict = True,
            return_tensors = "pt",
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            do_sample = False,
            max_new_tokens = 300,
            temperature = 0.01,
            repetition_penalty = 1.05
        )

        # Strip noise from text
        answer = re.sub(r"<think>.*?</think>", "", tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens = True), flags = re.DOTALL).strip()
        # if generation got cut off mid-thought, drop the unclosed block too
        answer = re.sub(r"<think>.*", "", answer, flags=re.DOTALL).strip()

    # Responds with error if model can't generate a response
    except Exception as e:
        st.error("The model couldn't generate a response. Please try again.")
        answer = "The model couldn't generate a response. Please try again."

    # returns model output
    return answer



# Provides user a seamless way to interact with the system through Streamlit
def deployToSt():

    Settings.embed_model, tokenizer, model = loadModel()

    st.set_page_config(page_title = "Cricket Knowledge Assistant")
    st.markdown("""
    <style>
    h2 {
        color: #FFFFFF;
    }
    </style>""", unsafe_allow_html=True)

    st.header("Welcome to the Cricket Knowledge Assistant!")

    with st.sidebar:

        st.subheader("📋 What I can answer:")
        st.markdown("""
        This assistant only has knowledge from these Wikipedia pages:

        ● **Rules & Formats**
        Cricket, Laws of Cricket, Test cricket, One Day International,
        ICC Men's T20 World Cup, Duckworth–Lewis–Stern method

        ● **Tournaments**
        Cricket World Cup, Indian Premier League

        ● **Players**
        Sachin Tendulkar, Virat Kohli, Jasprit Bumrah,
        Shane Warne, Ravichandran Ashwin

        Questions outside these topics (other sports, current events,
        general knowledge) won't be answered reliably — I'll say so if the
        context doesn't cover it.

        Example questions have been provided below. Feel free to copy and paste into the chat box:
        """)

        st.caption("Example 1: What is the Duckworth-Lewis-Stern method used for?")
        st.caption("Example 2: How many players are on each side in a game of cricket?")
        st.caption("Example 3: Who did India play in the final of the 2011 Cricket World Cup?")


    # --- ADMIN SIDEBAR FOR DYNAMIC KNOWLEDGE UPDATES ---
    # st.sidebar.header("Knowledge Management")
    # newPage = st.sidebar.text_input("Add Wikipedia Page (e.g., Jasprit Bumrah)", "")


    # Stores data from Wikipedia pages as Vector embeddings for retrieval. Gives index to access data in Pinecone
    idx, _ = loadData()


    # Dictionary of past questions (keys) and LLM answers (values)
    history = st.session_state.setdefault("History", {})
    chat = st.chat_input("Ask about cricket rules, players, or tournaments...")

    # If chat has been sent, will run the following
    if chat:

        # Typing "clear" wipes the conversation
        if chat.strip().lower() == "clear":
            history.clear()

        # Will run this IF user doesn't type "clear"
        else:

            with st.spinner("Thinking..."):

                # Otherwise will retrieve context from Pinecone database, send to inference model (Qwen 3) for response using RAG
                try:
                    retrievedData = idx.as_retriever(similarity_top_k = 5, embed_model = Settings.embed_model).retrieve(chat)
                    context = "\n\n".join([node.get_text() for node in retrievedData])
                    # Stores chat in Streamlit container calling llmInference passing in user query and wikipedia context
                    storeChat(chat, llmInference(tokenizer, model, chat, context))

                except Exception as e:
                    storeChat(chat, "I'm having trouble reaching the knowledge database right now. Please try again.")


    # Displays the whole conversation, oldest first, on every rerun
    for question, answer in history.items():

        # if isinstance(entry, str):
        #     entry = {"answer": entry, "sources": []}
        text_bubble(f"You: {question}", "#1F2022", "left", "#FFFFFF")
        text_bubble(f"Qwen: {answer['answer']}", "#53565A", "right", "#FFFFFF")


# -------------------------------------------------------------------
# Main function (calls load model and runs on Streamlit)
# -------------------------------------------------------------------

if __name__ == "__main__":

    # Initializes models, configures device, and settings to allow llamaindex to use the initialized models automatically
    # model_id = "Irfanuruchi/Qwen3-4B-Computer-Science-OpenVINO-INT4"
    # model_path = "qwen3-4b-cs-int4-ov"

    # Deploys project to Streamlit to communicate with the chatbot
    deployToSt()

# Run Evaluation

In [ ]:
# ===================================================================
# CLAUDE-GRADED RAG EVALUATION
# ===================================================================
%pip install -q anthropic
import anthropic
import pandas as pd
import importlib
import rag_chatbot
importlib.reload(rag_chatbot)
from rag_chatbot import loadModel, loadData, llmInference
from llama_index.core import Settings
from google.colab import files
from datetime import datetime


# Initializes variables to get access to models, database index.
Settings.embed_model, tokenizer, model = loadModel()
idx, vectorStore = loadData()

# API key: try Colab secrets first. If that's still giving you trouble,
# you can temporarily paste a key directly below for testing ONLY —
# just make sure to remove it before sharing/committing this notebook.
# try:
#     ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
# except Exception:
#     ANTHROPIC_API_KEY = None  # <- paste a key here temporarily if needed, e.g. "sk-ant-..."

aiJudge = anthropic.Anthropic(api_key = 'INSERT_YOUR_API_KEY_HERE')


# 50 questions store in list to provide to Qwen to test it's RAG capabilities.
eval_dataset = [
    # --- Rules ---
    {"question": "What happens if a bowler bowls a no-ball?",
     "expected_sources": ["wiki_laws_of_cricket"]},
    {"question": "How is a batter given out 'leg before wicket' (LBW)?",
     "expected_sources": ["wiki_laws_of_cricket"]},
    {"question": "What is the maximum number of fielders allowed outside the 30-yard circle in an ODI powerplay?",
     "expected_sources": ["wiki_one_day_international", "wiki_laws_of_cricket"]},
    {"question": "How many players are on each side in a game of cricket?",
     "expected_sources": ["wiki_cricket"]},
    {"question": "What is the standard length of a cricket pitch?",
     "expected_sources": ["wiki_cricket", "wiki_laws_of_cricket"]},
    {"question": "What does it mean for a batter to be given out 'run out'?",
     "expected_sources": ["wiki_laws_of_cricket"]},
    {"question": "What does it mean for a batter to be 'stumped'?",
     "expected_sources": ["wiki_laws_of_cricket"]},
    {"question": "What is the consequence of a batter hitting the ball over the boundary without it touching the ground?",
     "expected_sources": ["wiki_laws_of_cricket", "wiki_cricket"]},
    {"question": "How many dismissals are required to end an innings in a standard match?",
     "expected_sources": ["wiki_laws_of_cricket", "wiki_cricket"]},
    {"question": "What is a 'free hit' in white-ball cricket?",
     "expected_sources": ["wiki_laws_of_cricket", "wiki_one_day_international"]},

    # --- Tournament / Format Facts ---
    {"question": "Which country won the first Cricket World Cup, in 1975?",
     "expected_sources": ["wiki_cricket_world_cup"]},
    {"question": "Who did India play in the final of the 2011 Cricket World Cup?",
     "expected_sources": ["wiki_cricket_world_cup"]},
    {"question": "Who did India play in the semi-final of the 2011 Cricket World Cup?",
     "expected_sources": ["wiki_cricket_world_cup"]},
    {"question": "What is the Duckworth–Lewis–Stern method used for?",
     "expected_sources": ["wiki_duckworth-lewis-stern_method"]},
    {"question": "How many overs are bowled per side in a T20 International?",
     "expected_sources": ["wiki_icc_men's_t20_world_cup", "wiki_cricket"]},
    {"question": "In which year was the IPL founded?",
     "expected_sources": ["wiki_indian_premier_league"]},
    {"question": "How often is the Cricket World Cup held?",
     "expected_sources": ["wiki_cricket_world_cup"]},
    {"question": "Which country has won the most Cricket World Cup titles?",
     "expected_sources": ["wiki_cricket_world_cup"]},
    {"question": "When was the first ICC Men's T20 World Cup held?",
     "expected_sources": ["wiki_icc_men's_t20_world_cup"]},
    {"question": "Which country won the inaugural ICC Men's T20 World Cup in 2007?",
     "expected_sources": ["wiki_icc_men's_t20_world_cup"]},
    {"question": "How many days does a Test cricket match typically last?",
     "expected_sources": ["wiki_test_cricket"]},
    {"question": "What color ball is traditionally used in Test cricket?",
     "expected_sources": ["wiki_test_cricket"]},
    {"question": "How many overs are played per side in a One Day International match?",
     "expected_sources": ["wiki_one_day_international"]},
    {"question": "Why was the Duckworth-Lewis method originally created?",
     "expected_sources": ["wiki_duckworth-lewis-stern_method"]},
    {"question": "How many teams currently compete in the Indian Premier League?",
     "expected_sources": ["wiki_indian_premier_league"]},
    {"question": "What format of cricket is played in the IPL?",
     "expected_sources": ["wiki_indian_premier_league", "wiki_cricket"]},
    {"question": "Which franchise won the inaugural season of the Indian Premier League in 2008?",
     "expected_sources": ["wiki_indian_premier_league", "wiki_shane_warne_(cricketer)"]},
    {"question": "In what decade was the first Test match played?",
     "expected_sources": ["wiki_test_cricket", "wiki_cricket"]},

    # --- Player Profiles ---
    {"question": "What is Virat Kohli's role on the field (batter, bowler, or both)?",
     "expected_sources": ["wiki_virat_kohli"]},
    {"question": "Which format of cricket did Shane Warne primarily play?",
     "expected_sources": ["wiki_shane_warne_(cricketer)"]},
    {"question": "What nationality is Ravichandran Ashwin?",
     "expected_sources": ["wiki_ravichandran_ashwin"]},
    {"question": "What is Jasprit Bumrah best known for as a bowler?",
     "expected_sources": ["wiki_jasprit_bumrah"]},
    {"question": "How many international centuries did Sachin Tendulkar score in his career?",
     "expected_sources": ["wiki_sachin_tendulkar"]},
    {"question": "What is Sachin Tendulkar's commonly used nickname?",
     "expected_sources": ["wiki_sachin_tendulkar"]},
    {"question": "Has Virat Kohli ever captained the Indian national cricket team?",
     "expected_sources": ["wiki_virat_kohli"]},
    {"question": "What is unusual or notable about Jasprit Bumrah's bowling action?",
     "expected_sources": ["wiki_jasprit_bumrah"]},
    {"question": "How many Test wickets did Shane Warne take over his career?",
     "expected_sources": ["wiki_shane_warne_(cricketer)"]},
    {"question": "Is Ravichandran Ashwin a spin bowler or a pace bowler?",
     "expected_sources": ["wiki_ravichandran_ashwin"]},
    {"question": "Which IPL team did Shane Warne captain to a title victory?",
     "expected_sources": ["wiki_shane_warne_(cricketer)", "wiki_indian_premier_league"]},
    {"question": "In what year did Sachin Tendulkar make his international debut?",
     "expected_sources": ["wiki_sachin_tendulkar"]},
    {"question": "Has Jasprit Bumrah ever taken a hat-trick in Test cricket?",
     "expected_sources": ["wiki_jasprit_bumrah"]},
    {"question": "Which bowling style is Ravichandran Ashwin known for using as his primary delivery?",
     "expected_sources": ["wiki_ravichandran_ashwin"]},
    {"question": "In which city or state was Virat Kohli born?",
     "expected_sources": ["wiki_virat_kohli"]},

    # --- Out-of-Scope / Expected Refusals ---
    {"question": "Who holds the record for most Test wickets by a spinner, Muttiah Muralitharan or Shane Warne?",
     "expected_sources": []},  # Requires Muralitharan context
    {"question": "What is the offside rule in football?",
     "expected_sources": []},  # Non-cricket
    {"question": "Who won the most recent Cricket World Cup?",
     "expected_sources": []},  # Temporal ambiguity outside corpus
    {"question": "How many total Test wickets did Muttiah Muralitharan take in his career?",
     "expected_sources": []},  # Unindexed player
    {"question": "Who holds the record for the fastest goal scored in FIFA World Cup history?",
     "expected_sources": []},  # Non-cricket
    {"question": "Who won the FIFA Women's World Cup in 2023?",
     "expected_sources": []},  # Non-cricket
    {"question": "What is the career high score of Brian Lara in Test cricket?",
     "expected_sources": []}   # Unindexed player
]


# Sends question from dataset, wikipedia context, and Qwen's answer to Claude API to receive grade
def judgeAnswer(question, context, generated_answer):
    """Asks Claude to grade one RAG answer for correctness and evidence support."""

    # Judge prompt takes in the appropriate parameters formatted into a robust prompt which will be used to grade Qwen's outputs (LLM-as-judge)
    judgePrompt = f"""
    You are grading a RAG system's answer for factual accuracy.

    <question>
    {question}
    </question>

    <retrieved_context>
    {context}
    </retrieved_context>

    <model_answer>
    {generated_answer}
    </model_answer>

    Check:
    1. Is the claim in the answer factually correct?
    2. If the answer includes an "Evidence:" quote, does that quote actually appear in the
    retrieved context and actually support the claim made (not just a nearby, related fact)?
    3. If the retrieved context does not contain the answer, did the model correctly say so
    rather than guessing?

    Respond with exactly one word on the first line: CORRECT, INCORRECT, or UNSUPPORTED
    (meaning the evidence doesn't back the claim, even if the claim happens to be true).
    Then one short sentence explaining why.
    """

    # Takes in system prompt using Claude Sonnet model to provide inference
    response = aiJudge.messages.create(
        model = "claude-sonnet-5-5",
        max_tokens = 1024,
        # extra_body={"temperature": 0.0},
        # output_config = {"effort": "low"},
        messages = [{"role": "user", "content": judgePrompt}]
    )

    final_text = "".join(block.text for block in response.content if block.type == "text")
    return final_text.strip()


# Claude provides inference and Python code is used to manually calculate a score for whether Qwen got it CORRECT, INCORRECT, or UNSUPPORTED (If it can't retrieve certain data)
def run_claude_graded_evaluation(idx, tokenizer, model):
    results_log = []
    total_retrieval_hits = 0
    total_judge_correct = 0

    for i, run in enumerate(eval_dataset, 1):
        q = run["question"]

        # 1. Retrieval check (same deterministic approach as before)
        retrieved_nodes = idx.as_retriever(similarity_top_k = 5).retrieve(q)
        active_source_ids = [node.node.ref_doc_id for node in retrieved_nodes if node.node.ref_doc_id]

        if run["expected_sources"]:
            retrieval_hit = any(src in active_source_ids for src in run["expected_sources"])
        else:
            # out-of-scope questions "pass" retrieval if nothing confidently relevant came back
            retrieval_hit = True

        if retrieval_hit:
            total_retrieval_hits += 1

        # 2. Generate the answer the same way your app does
        context_str = "\n\n".join([node.get_text() for node in retrieved_nodes])
        generated_answer = llmInference(tokenizer, model, q, context_str)

        # 3. Claude judges correctness + evidence support
        verdict = judgeAnswer(q, context_str, generated_answer)
        verdict_label = verdict.splitlines()[0].strip() if verdict.strip() else "UNKNOWN"
        is_correct = verdict_label.startswith("CORRECT")
        if is_correct:
            total_judge_correct += 1

        results_log.append({
            "No.": i,
            "Question": q[:50] + "...",
            "Retrieval Hit": "🟢" if retrieval_hit else "🔴",
            "Judge Verdict": verdict_label,
            "Judge Reasoning": " ".join(verdict.splitlines()[1:])[:100]
        })

    total_q = len(eval_dataset)
    df = pd.DataFrame(results_log)

    print("\n" + "=" * 90)
    print("          CLAUDE-GRADED RAG EVALUATION REPORT")
    print("=" * 90)
    print(df.to_string(index = False))
    print("-" * 90)
    print(f"Retrieval Hit Rate:     {total_retrieval_hits}/{total_q} ({total_retrieval_hits/total_q*100:.1f}%)")
    print(f"Judge-Verified Correct: {total_judge_correct}/{total_q} ({total_judge_correct/total_q*100:.1f}%)")
    print("=" * 90)
    print([node.node.ref_doc_id for node in retrieved_nodes])

    return df


# Exports results to excel sheet for ease of viewing results
def exportResults(df):
    """Appends summary stats to the eval results and downloads CSV + Excel copies."""

    total_q = len(df)
    retrieval_hits = (df["Retrieval Hit"] == "🟢").sum()
    judge_correct = df["Judge Verdict"].str.startswith("CORRECT").sum()

    summaryRow = pd.DataFrame([{
        "No.": "",
        "Question": "SUMMARY",
        "Retrieval Hit": f"{retrieval_hits}/{total_q} ({retrieval_hits/total_q*100:.1f}%)",
        "Judge Verdict": f"{judge_correct}/{total_q} ({judge_correct/total_q*100:.1f}%)",
        "Judge Reasoning": ""
    }])

    exportDf = pd.concat([df, summaryRow], ignore_index = True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csvName = f"rag_eval_results_{timestamp}.csv"
    xlsxName = f"rag_eval_results_{timestamp}.xlsx"

    exportDf.to_csv(csvName, index = False)
    exportDf.to_excel(xlsxName, index = False)

    print(f"Retrieval Hit Rate:     {retrieval_hits}/{total_q} ({retrieval_hits/total_q*100:.1f}%)")
    print(f"Judge-Verified Correct: {judge_correct}/{total_q} ({judge_correct/total_q*100:.1f}%)")

    files.download(csvName)
    files.download(xlsxName)

    return exportDf


# Main function calls Claude evaluation first then returns Pandas dataframe which is used to download results
if __name__ == "__main__":

    df = run_claude_graded_evaluation(idx, tokenizer, model)
    exportResults(df)

# Launch Streamlit

In [ ]:
# Download cloudflared binary and make it executable
!wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
# =========================================================
# Cell 2: Launch Streamlit + expose via Cloudflare tunnel
# =========================================================
import sys
import time

# 1. Clean up any hanging background processes from previous runs
!pkill -9 -f streamlit 2>/dev/null
!pkill -9 -f cloudflared 2>/dev/null
!rm -f streamlit.log

# 2. Ensure Streamlit is installed in the active Python environment
!{sys.executable} -m pip install -q streamlit

# 3. Start Streamlit in background using active Python executable
get_ipython().system_raw(
    f"{sys.executable} -m streamlit run rag_chatbot.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &"
)

# 4. Wait for Streamlit to start up
time.sleep(5)

print("--- STREAMLIT BOOT LOG ---")
!tail -30 streamlit.log
print("--------------------------\n")

print("=" * 75)
print("🚀 Look for the '.trycloudflare.com' URL in the output below.")
print("=" * 75 + "\n")

# 5. Launch Cloudflare Tunnel
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501